## Demo for OpenAlex

In [1]:
# Demo: OpenAlex API for Best-Paper Award Analysis
# Author: Shaheryar
# Date: January 2026

import requests
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from collections import defaultdict
import time

# Set style
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (10, 6)

print("✓ Libraries loaded")


✓ Libraries loaded


In [ ]:
def get_paper_by_doi(doi):
    """Retrieve paper by DOI (most reliable method)."""
    url = f"https://api.openalex.org/works/https://doi.org/{doi}"
    response = requests.get(url)
    
    if response.status_code == 200:
        return response.json()
    return None


def search_openalex_paper(title=None, year=None, venue_id=None, doi=None):
    """
    Search for a paper in OpenAlex.
    Prefer DOI when available, fall back to title search.
    """
    # Method 1: DOI (most reliable)
    if doi:
        return get_paper_by_doi(doi)
    
    # Method 2: Title search (less reliable)
    if not title:
        raise ValueError("Must provide either DOI or title")
    
    url = "https://api.openalex.org/works"
    params = {"search": title}
    
    filters = []
    if year:
        filters.append(f"publication_year:{year}")
    if venue_id:
        filters.append(f"primary_location.source.id:{venue_id}")
    
    if filters:
        params["filter"] = ",".join(filters)
    
    response = requests.get(url, params=params)
    results = response.json().get('results', [])
    return results[0] if results else None


def get_author_works(author_id):
    """Retrieve all works for a given author (with pagination)."""
    all_works = []
    cursor = "*"
    
    while cursor:
        url = "https://api.openalex.org/works"
        params = {
            "filter": f"author.id:{author_id}",
            "per_page": 200,
            "cursor": cursor,
            "sort": "publication_year:asc"
        }
        
        response = requests.get(url, params=params)
        data = response.json()
        
        results = data.get('results', [])
        all_works.extend(results)
        
        # Get next cursor for pagination
        cursor = data.get('meta', {}).get('next_cursor')
        
        time.sleep(0.05)  # Polite rate limiting
    
    return all_works



def calculate_career_age(works, reference_year):
    """Calculate years since first publication."""
    years = [w['publication_year'] for w in works if w.get('publication_year')]
    if not years:
        return None
    first_year = min(years)
    return reference_year - first_year

print("✓ Helper functions defined")

✓ Helper functions defined


In [6]:
# Better approach: Search CHI venue specifically
# First, let's verify the CHI venue ID

url = "https://api.openalex.org/sources"
params = {"search": "CHI Conference on Human Factors"}
response = requests.get(url, params=params)
sources = response.json()['results']

print("CHI venue options:")
for i, source in enumerate(sources[:3], 1):
    print(f"{i}. {source['display_name']}")
    print(f"   ID: {source['id']}")
    print(f"   Type: {source.get('type', 'N/A')}")
    print()


CHI venue options:
1. CHI Conference on Human Factors in Computing Systems
   ID: https://openalex.org/S4363607743
   Type: conference

2. CHI Conference on Human Factors in Computing Systems Extended Abstracts
   ID: https://openalex.org/S4363607762
   Type: conference



In [11]:
# CHI 2020 Best Papers with DOIs
# Source: ACM Digital Library + https://chi2020.acm.org/for-attendees/chi-2020-best-papers-honourable-mentions/

chi_2020_best_papers_dois = [
    {
        'title': 'PenSight: Enhanced Interaction with a Pen-Top Camera',
        'doi': '10.1145/3313831.3376147'
    },
    {
        'title': 'Robots for Inclusive Play: Co-designing an Educational Game',
        'doi': '10.1145/3313831.3376270'
    },
    {
        'title': 'texSketch: Active Diagramming through Pen-and-Ink Annotations',
        'doi': '10.1145/3313831.3376155'
    }
]

print(f"Querying {len(chi_2020_best_papers_dois)} CHI 2020 Best Papers via DOI...")
print("=" * 70)

found_papers = []

for paper in chi_2020_best_papers_dois:
    # Query by DOI directly (NOT by search!)
    url = f"https://api.openalex.org/works/https://doi.org/{paper['doi']}"
    
    response = requests.get(url)
    
    if response.status_code == 200:
        work = response.json()
        found_papers.append(work)
        print(f"✓ {paper['title'][:60]}...")
        print(f"  OpenAlex ID: {work['id']}")
        print(f"  Citations: {work['cited_by_count']} | Authors: {len(work['authorships'])}")
    else:
        print(f"✗ NOT FOUND (HTTP {response.status_code}): {paper['title'][:60]}...")
    
    time.sleep(0.1)  # Polite API usage

print(f"\n✓ Successfully found {len(found_papers)}/{len(chi_2020_best_papers_dois)} papers")

# Display first paper details
if found_papers:
    print("\n" + "=" * 70)
    print("FIRST PAPER DETAILS:")
    print("=" * 70)
    work = found_papers[0]
    print(f"Title: {work['display_name']}")
    print(f"Year: {work['publication_year']}")
    print(f"Venue: {work.get('primary_location', {}).get('source', {}).get('display_name')}")
    print(f"Citations: {work['cited_by_count']}")

Querying 3 CHI 2020 Best Papers via DOI...
✓ PenSight: Enhanced Interaction with a Pen-Top Camera...
  OpenAlex ID: https://openalex.org/W3029640333
  Citations: 31 | Authors: 4
✓ Robots for Inclusive Play: Co-designing an Educational Game...
  OpenAlex ID: https://openalex.org/W3030686314
  Citations: 103 | Authors: 5
✓ texSketch: Active Diagramming through Pen-and-Ink Annotation...
  OpenAlex ID: https://openalex.org/W3029893033
  Citations: 38 | Authors: 4

✓ Successfully found 3/3 papers

FIRST PAPER DETAILS:
Title: PenSight: Enhanced Interaction with a Pen-Top Camera
Year: 2020


AttributeError: 'NoneType' object has no attribute 'get'